# DevDas — Autonomous Kubernetes SRE Investigation Agent

This notebook consolidates the full DevDas project into a single runnable file.

**Run cells top to bottom.** Order matches the import dependency chain:
`llm` → `kubernetes_tools` → `ask_llm` → `evidence` → `tools` → `system_prompt` → `discovery` → `investigate` → `report` → `main`

> ℹ️ **Model note:** the original `llm.py` called the Groq API with a hardcoded key. It's replaced below with a **locally-loaded Qwen2.5-3B-Instruct model** via  `transformers` 


## Setup — install dependencies

Installs the packages needed to load and run the local Qwen model (`transformers`, `torch`, `accelerate`), plus `requests` for talking to the Kubernetes tool server over ngrok.


## `llm.py`
Loads the **Qwen2.5-3B-Instruct** tokenizer and model locally via `transformers` (replaces the old Groq API client). This runs once and keeps `tokenizer`/`model` in memory for the rest of the notebook — everything downstream only ever calls `ask_llm()`, so it doesn't matter to them that inference is now local instead of an API call.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
# import os
# from groq import Groq

# client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY", "PASTE_YOUR_GROQ_API_KEY_HERE")
# )

# MODEL_NAME = "llama-3.3-70b-versatile"


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

## `kubernetes_tools.py`
`kubectl`-backed cluster lives on your **local machine**, not here — so instead of shelling out to `kubectl` directly, this notebook talks to a local Kubernetes Tool Server (FastAPI, running on your machine and exposed via ngrok) over HTTP.

Every function below keeps the **exact same name and signature** as the original `kubernetes_tools.py`, so `tools.py`, `discovery.py`, and `investigate.py` further down don't need any changes — they just call these functions, which forward to `POST {NGROK_BASE_URL}/execute_tool` under the hood. Independent of the Qwen model change above.

> Make sure your local FastAPI tool server (the one wrapping `kubernetes_tools`) is running and the ngrok tunnel is live before executing investigations.


In [2]:
"""
kubernetes_tools.py  (remote client version)

Instead of running `kubectl` locally (this notebook has no access to your
cluster), every function here forwards to a FastAPI tool server running on
your local machine, exposed through ngrok. The server wraps the *real*
kubernetes_tools.py and executes the actual kubectl commands.

Each function below preserves the exact name + signature of the original
local functions, so nothing downstream (tools.py, discovery.py,
investigate.py) needs to change.
"""

import requests

# ---------------------------------------------------------------------------
# Config — your ngrok tunnel to the local DevDas Kubernetes Tool Server
# ---------------------------------------------------------------------------

NGROK_BASE_URL = "https://excavate-dreadlock-gumminess.ngrok-free.dev"
REQUEST_TIMEOUT = 60  # seconds


def _call_remote_tool(tool_name: str, **arguments):
    """
    POST to the local tool server's /execute_tool endpoint and return
    the tool's result. Raises on network failure or a remote error, so
    callers (tools.py's execute_tool, discovery.py, investigate.py) see
    the same exception-based error handling they had with local kubectl.
    """
    # Drop None-valued args so server-side defaults kick in
    payload_args = {k: v for k, v in arguments.items() if v is not None}

    try:
        resp = requests.post(
            f"{NGROK_BASE_URL}/execute_tool",
            json={"tool": tool_name, "arguments": payload_args},
            timeout=REQUEST_TIMEOUT,
            headers={"ngrok-skip-browser-warning": "true"},
        )
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(
            f"[DevDas] Failed to reach tool server at {NGROK_BASE_URL} "
            f"for tool '{tool_name}': {e}"
        )

    data = resp.json()

    if not data.get("success", False):
        raise RuntimeError(
            f"[DevDas] Remote tool '{tool_name}' failed: {data.get('error', 'unknown error')}"
        )

    return data["result"]


# ---------------------------------------------------------------------------
# Discovery (still "pure" from the notebook's perspective — the actual
# kubectl scan happens on the server side inside discover_unhealthy_workloads)
# ---------------------------------------------------------------------------

def discover_unhealthy_workloads(namespace: str = "--all-namespaces"):
    """
    Discover all unhealthy pods in the cluster via the remote tool server.
    Returns (unhealthy_workloads: list[dict], total_pods: int) — same shape
    as the local version. The server returns this as a two-element list;
    we coerce it back to a tuple.
    """
    result = _call_remote_tool("discover_unhealthy_workloads", namespace=namespace)
    unhealthy, total_pods = result
    return unhealthy, total_pods


def resolve_deployment_name(pod_name: str, namespace: str):
    """Walk Pod -> ReplicaSet -> Deployment to find the real deployment name."""
    return _call_remote_tool("resolve_deployment_name", pod_name=pod_name, namespace=namespace)


def get_all_pods(namespace: str = "--all-namespaces") -> str:
    """Return a human-readable pod listing across the cluster."""
    return _call_remote_tool("get_all_pods", namespace=namespace)


# ---------------------------------------------------------------------------
# Pod-level tools
# ---------------------------------------------------------------------------

def get_pod_logs(pod_name: str, namespace: str, tail: int = 100, previous: bool = False) -> str:
    """Fetch logs from a pod. Use previous=True for the previously crashed container."""
    return _call_remote_tool(
        "get_pod_logs", pod_name=pod_name, namespace=namespace, tail=tail, previous=previous
    )


def get_pod_events(pod_name: str, namespace: str) -> str:
    """Fetch Kubernetes events for a specific pod."""
    return _call_remote_tool("get_pod_events", pod_name=pod_name, namespace=namespace)


def describe_pod(pod_name: str, namespace: str) -> dict:
    """Return a structured dict with key pod details."""
    return _call_remote_tool("describe_pod", pod_name=pod_name, namespace=namespace)


# ---------------------------------------------------------------------------
# Deployment-level tools
# ---------------------------------------------------------------------------

def describe_deployment(deployment_name: str, namespace: str) -> dict:
    """Return a structured dict with deployment diagnostics."""
    return _call_remote_tool(
        "describe_deployment", deployment_name=deployment_name, namespace=namespace
    )


def get_deployment_yaml(deployment_name: str, namespace: str) -> str:
    """Get the raw YAML spec of a deployment for deep inspection."""
    return _call_remote_tool(
        "get_deployment_yaml", deployment_name=deployment_name, namespace=namespace
    )


def get_replicaset_status(namespace: str) -> str:
    """List all ReplicaSets in a namespace to identify rollout issues."""
    return _call_remote_tool("get_replicaset_status", namespace=namespace)


# ---------------------------------------------------------------------------
# Service & networking tools
# ---------------------------------------------------------------------------

def describe_service(service_name: str, namespace: str) -> str:
    """Describe a Kubernetes service — endpoints, selector, ports."""
    return _call_remote_tool("describe_service", service_name=service_name, namespace=namespace)


def get_endpoints(service_name: str, namespace: str) -> str:
    """Get endpoints for a service — useful for diagnosing connectivity issues."""
    return _call_remote_tool("get_endpoints", service_name=service_name, namespace=namespace)


# ---------------------------------------------------------------------------
# Configuration tools
# ---------------------------------------------------------------------------

def get_configmap(configmap_name: str, namespace: str) -> str:
    """Fetch a ConfigMap's data."""
    return _call_remote_tool("get_configmap", configmap_name=configmap_name, namespace=namespace)


def get_secret(secret_name: str, namespace: str) -> str:
    """Fetch a Secret — shows key NAMES only, values are redacted server-side."""
    return _call_remote_tool("get_secret", secret_name=secret_name, namespace=namespace)


# ---------------------------------------------------------------------------
# Node tools
# ---------------------------------------------------------------------------

def get_nodes() -> str:
    """List all cluster nodes with their status and roles."""
    return _call_remote_tool("get_nodes")


def describe_node(node_name: str) -> str:
    """Describe a node — resource pressure, taints, and conditions."""
    return _call_remote_tool("describe_node", node_name=node_name)


# ---------------------------------------------------------------------------
# Namespace & cluster-wide tools
# ---------------------------------------------------------------------------

def get_namespace_resources(namespace: str) -> str:
    """List all resources in a namespace — deployments, services, configmaps, etc."""
    return _call_remote_tool("get_namespace_resources", namespace=namespace)


def get_persistent_volume_claims(namespace: str) -> str:
    """List PVCs in a namespace — useful for storage-related failures."""
    return _call_remote_tool("get_persistent_volume_claims", namespace=namespace)


def get_resource_quota(namespace: str) -> str:
    """Check resource quotas and limits in a namespace."""
    return _call_remote_tool("get_resource_quota", namespace=namespace)


def get_horizontal_pod_autoscaler(namespace: str) -> str:
    """Check HPA status — useful for scaling-related investigations."""
    return _call_remote_tool("get_horizontal_pod_autoscaler", namespace=namespace)


# ---------------------------------------------------------------------------
# Quick sanity check against the tool server
# ---------------------------------------------------------------------------

def _check_tool_server():
    try:
        health = requests.get(f"{NGROK_BASE_URL}/", timeout=10,
                               headers={"ngrok-skip-browser-warning": "true"}).json()
        tools = requests.get(f"{NGROK_BASE_URL}/tools", timeout=10,
                              headers={"ngrok-skip-browser-warning": "true"}).json()
        print(f"[DevDas] Tool server reachable: {health}")
        print(f"[DevDas] {len(tools.get('tools', []))} tools available on server.")
    except Exception as e:
        print(f"[DevDas] WARNING: could not reach tool server at {NGROK_BASE_URL}: {e}")

_check_tool_server()


[DevDas] Tool server reachable: {'status': 'running', 'service': 'DevDas Tool Server'}
[DevDas] 19 tools available on server.


## `ask_llm.py`
Thin wrapper around local Qwen inference: builds the chat-formatted prompt with `tokenizer.apply_chat_template`, generates a completion with `model.generate()`, decodes only the newly-generated tokens, and strips markdown code fences from the result. The retry/backoff loop is preserved from the Groq version but is mostly dormant now — there's no external API rate limit to hit locally.


In [3]:
import time as _time

def ask_llm(system_prompt, user_prompt, max_retries=5):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    for attempt in range(max_retries):
        try:
            input_ids = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(model.device)

            output_ids = model.generate(
                input_ids,
                max_new_tokens=1024,
                temperature=0.2,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

            generated_ids = output_ids[0][input_ids.shape[-1]:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            break
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  [RATE LIMIT] Retrying in {wait}s...")
                _time.sleep(wait)
                continue
            raise

    if response.startswith("```json"):
        response = response[7:]

    if response.startswith("```"):
        response = response[3:]

    if response.endswith("```"):
        response = response[:-3]

    return response.strip()

## `evidence.py`
Per-workload evidence store factory — a fresh, isolated dict per investigation. Unchanged by the model swap.


In [4]:
def new_evidence() -> dict:
    """
    Return a fresh evidence dictionary for one workload investigation.
    Keys are added dynamically as tools execute — no pre-defined schema.
    """
    return {}


## `tools.py`
Tool routing layer: `AVAILABLE_TOOLS` (the schema shown to the LLM in its system prompt) + `execute_tool()`, which dispatches the LLM's chosen tool calls into `kubernetes_tools`. 


In [5]:
"""
tools.py

Tool routing layer.

Responsibilities:
    1. Define AVAILABLE_TOOLS — a dict the LLM uses to understand
       what tools exist and what arguments they accept.
    2. Implement execute_tool() — routes LLM tool requests to real
       kubernetes_tools functions and stores results in the caller's
       per-workload evidence dict.

Architecture:
    LLM response → execute_tool(tool, args, evidence)
                        ↓
                   kubernetes_tools.*()
                        ↓
                   subprocess.run("kubectl ...")
                        ↓
                   evidence[key] = result
"""

# NOTE: all kubernetes_tools functions are already defined above in this notebook,
# so no import is needed here.


# ---------------------------------------------------------------------------
# AVAILABLE_TOOLS
# Description shown to LLM so it knows what parameters to pass.
# ---------------------------------------------------------------------------

AVAILABLE_TOOLS = {
    "get_pod_logs": {
        "description": "Fetch recent stdout/stderr logs from a pod. "
                       "Use previous=true to get logs from a crashed container.",
        "arguments": {
            "pod_name": "string (required) — exact pod name",
            "namespace": "string (required) — Kubernetes namespace",
            "tail": "integer (optional, default 100) — number of log lines to fetch",
            "previous": "boolean (optional, default false) — fetch previous container logs"
        }
    },
    "get_pod_events": {
        "description": "Fetch Kubernetes events for a specific pod. "
                       "Reveals scheduling failures, image pull errors, OOMKilled, etc.",
        "arguments": {
            "pod_name": "string (required) — exact pod name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "describe_pod": {
        "description": "Return a structured JSON dict with key pod fields (phase, reason, image, imagePullPolicy, node, containerState, restartCount).",
        "arguments": {
            "pod_name": "string (required) — exact pod name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "describe_deployment": {
        "description": "Return a structured JSON dict with deployment diagnostics (image, imagePullPolicy, configmaps, secrets, services, replicas, readyReplicas).",
        "arguments": {
            "deployment_name": "string (required) — deployment name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_deployment_yaml": {
        "description": "Get the full YAML spec of a deployment. "
                       "Useful for inspecting resource limits, env vars, and volume mounts.",
        "arguments": {
            "deployment_name": "string (required) — deployment name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_replicaset_status": {
        "description": "List all ReplicaSets in a namespace. "
                       "Useful for identifying rollout or scaling issues.",
        "arguments": {
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "describe_service": {
        "description": "Describe a Kubernetes service — shows selector, ports, endpoints. "
                       "Use when investigating connectivity or traffic routing issues.",
        "arguments": {
            "service_name": "string (required) — service name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_endpoints": {
        "description": "Get endpoints for a service. "
                       "Use to verify whether pods are correctly registered as endpoints.",
        "arguments": {
            "service_name": "string (required) — service name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_configmap": {
        "description": "Fetch a ConfigMap's data. "
                       "Use when environment variable issues or missing config is suspected.",
        "arguments": {
            "configmap_name": "string (required) — ConfigMap name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_secret": {
        "description": "List a Secret's key names (values are redacted for security). "
                       "Use to verify whether expected secret keys exist.",
        "arguments": {
            "secret_name": "string (required) — Secret name",
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_nodes": {
        "description": "List all cluster nodes with status and roles. "
                       "Use when investigating node-level failures or scheduling issues.",
        "arguments": {}
    },
    "describe_node": {
        "description": "Describe a specific cluster node — taints, conditions, "
                       "resource pressure, and allocated resources.",
        "arguments": {
            "node_name": "string (required) — node name (from get_nodes output)"
        }
    },
    "get_namespace_resources": {
        "description": "List all resources (pods, services, deployments, etc.) "
                       "in a namespace. Use for a broad overview.",
        "arguments": {
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_persistent_volume_claims": {
        "description": "List PVCs in a namespace. "
                       "Use when storage-related failures are suspected.",
        "arguments": {
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_resource_quota": {
        "description": "Check resource quotas and usage in a namespace. "
                       "Use when pods fail to schedule due to resource limits.",
        "arguments": {
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
    "get_all_pods": {
        "description": "List all pods across the cluster (or a namespace) with wide output. "
                       "Use for a broad cluster-wide or namespace-wide pod overview.",
        "arguments": {
            "namespace": "string (optional, default '--all-namespaces') — Kubernetes namespace"
        }
    },
    "get_horizontal_pod_autoscaler": {
        "description": "Check HPA status — current vs desired replicas, metrics. "
                       "Use when autoscaling or replica count anomalies are suspected.",
        "arguments": {
            "namespace": "string (required) — Kubernetes namespace"
        }
    },
}


# ---------------------------------------------------------------------------
# Tool executor
# ---------------------------------------------------------------------------

def execute_tool(tool_name: str, args: dict, evidence: dict) -> str:
    result = _dispatch(tool_name, args)
    evidence[tool_name] = result
    return result


def _dispatch(tool_name: str, args: dict) -> str:
    """Route tool_name to the correct kubernetes_tools function."""

    # ── Pod tools ──────────────────────────────────────────────────────────
    if tool_name == "get_pod_logs":
        return get_pod_logs(
            pod_name=args.get("pod_name", ""),
            namespace=args.get("namespace", "default"),
            tail=int(args.get("tail", 100)),
            previous=bool(args.get("previous", False)),
        )

    elif tool_name == "get_pod_events":
        return get_pod_events(
            pod_name=args.get("pod_name", ""),
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "describe_pod":
        return describe_pod(
            pod_name=args.get("pod_name", ""),
            namespace=args.get("namespace", "default"),
        )

    # ── Deployment tools ───────────────────────────────────────────────────
    elif tool_name == "describe_deployment":
        return describe_deployment(
            deployment_name=(
                args.get("deployment_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_deployment_yaml":
        return get_deployment_yaml(
            deployment_name=(
                args.get("deployment_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_replicaset_status":
        return get_replicaset_status(
            namespace=args.get("namespace", "default"),
        )


    # ── Service tools ──────────────────────────────────────────────────────
    elif tool_name == "describe_service":
        return describe_service(
            service_name=(
                args.get("service_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_endpoints":
        return get_endpoints(
            service_name=(
                args.get("service_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )


    # ── Config tools ───────────────────────────────────────────────────────
    elif tool_name == "get_configmap":
        return get_configmap(
            configmap_name=(
                args.get("configmap_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_secret":
        return get_secret(
            secret_name=(
                args.get("secret_name")
                or args.get("name")
                or ""
            ),
            namespace=args.get("namespace", "default"),
        )


    # ── Node tools ─────────────────────────────────────────────────────────
    elif tool_name == "get_nodes":
        return get_nodes()

    elif tool_name == "describe_node":
        return describe_node(
            node_name=(
                args.get("node_name")
                or args.get("name")
                or ""
            ),
        )

    # ── Namespace / cluster tools ──────────────────────────────────────────
    elif tool_name == "get_namespace_resources":
        return get_namespace_resources(
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_persistent_volume_claims":
        return get_persistent_volume_claims(
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_resource_quota":
        return get_resource_quota(
            namespace=args.get("namespace", "default"),
        )

    elif tool_name == "get_all_pods":
        return get_all_pods(
            namespace=args.get("namespace", "--all-namespaces"),
        )

    elif tool_name == "get_horizontal_pod_autoscaler":
        return get_horizontal_pod_autoscaler(
            namespace=args.get("namespace", "default"),
        )

    else:
        return f"[DevDas error] Unknown tool: '{tool_name}'"

## `system_prompt.py`
The DevDas SRE persona/system prompt, built dynamically from `AVAILABLE_TOOLS`. 


In [6]:
"""
system_prompt.py

System prompt for DevDas — the autonomous Kubernetes SRE investigation agent.

The system prompt is designed to make the LLM behave like an experienced SRE:
- Evidence-driven reasoning only
- One tool at a time
- Eliminate hypotheses based on evidence
- Stop as soon as root cause is confident
- Return structured JSON at every step
"""

import json

# Build a compact tool listing string at import time
# (AVAILABLE_TOOLS is already defined above in this notebook)

_tool_list = "\n".join(
    f"  - {name}: {meta['description']}"
    for name, meta in AVAILABLE_TOOLS.items()
)

system_prompt = f"""
You are DevDas, an autonomous Kubernetes SRE investigation agent.

Your job is to determine the root cause of a Kubernetes workload failure
by gathering evidence ONE TOOL AT A TIME and reasoning step by step.

Your job is strictly: "Why is this workload unhealthy right now?"
You are NOT responsible for: why a resource was deleted, who deleted it,
historical root cause, or process/audit trail. Once the proximate
Kubernetes-level cause is confirmed, stop.

====================
GOLDEN RULES
====================

1. Never invent evidence. Only reason from what you have actually observed.
2. Never assume missing information. If a key is missing, that IS evidence.
3. Request EXACTLY ONE tool per response when more evidence is needed.
4. Only use tool names from the AVAILABLE TOOLS list.
5. Never request a tool that already appears in "tools_used".
6. Never output kubectl commands or shell commands.
7. Return ONLY valid JSON — no markdown, no prose, no code fences.
8. The pod_name and namespace you were given at the start are your starting point.
9. If a tool result starts with "SYSTEM_ERROR", that tool FAILED. Do NOT retry it.
   Pick a DIFFERENT tool that has not yet been used.

====================
AVAILABLE TOOLS
====================

{_tool_list}

Tool arguments are shown in the output schema below.
Always include all required arguments in next_action.arguments.

====================
TRIAGE GUIDE — fastest path by error status
====================

CreateContainerConfigError  →  Missing or misconfigured ConfigMap / Secret.
  Recommended order: get_pod_events → describe_deployment → get_configmap / get_secret

CrashLoopBackOff            →  Application is crashing repeatedly.
  Recommended order: get_pod_logs(previous=true) → get_pod_events

ImagePullBackOff / ErrImagePull  →  Bad image name or missing image pull secret.
  Recommended order: get_pod_events → describe_deployment

OOMKilled                   →  Container exceeded its memory limit.
  Recommended order: get_pod_logs(previous=true) → get_deployment_yaml

Phase:Pending               →  Pod cannot be scheduled (resources, taints, PVC).
  Recommended order: get_pod_events → get_nodes → get_resource_quota

====================
INVESTIGATION PROCESS
====================

At every step:

1. Read the workload description (pod name, namespace, status).
2. Read ALL collected evidence carefully.
3. Review the tools_used list — never re-request a used tool.
4. Form a ranked list of hypotheses (most likely first).
5. Eliminate hypotheses that are contradicted by existing evidence.
6. Ask: "Does the remaining evidence uniquely point to one root cause?"
   - If YES → return status: "complete" immediately.
   - If NO  → pick the ONE tool that eliminates the MOST remaining hypotheses.

====================
STOPPING RULES
====================

Stop and return "complete" when ANY of these are true:
- One hypothesis explains ALL collected evidence.
- Competing hypotheses have been eliminated by evidence.
- A production SRE could act on your diagnosis right now.
- All remaining unused tools would return redundant information.
- Tools are failing (SYSTEM_ERROR) and no alternative tool exists to advance the investigation.

Do NOT:
- Request a tool just because it hasn't been used.
- Continue investigating after the root cause is clear.
- Investigate healthy infrastructure when the problem is clearly in the app config.
- Retry any tool that previously returned SYSTEM_ERROR.

====================
TOOL SELECTION DISCIPLINE
====================

Before picking a tool, ask yourself:
1. Which hypothesis is currently most likely?
2. Which tool would most directly confirm OR eliminate it?
3. Is there any tool remaining that would change my conclusion?

If no remaining tool would change your conclusion → stop immediately.

====================
OUTPUT SCHEMA
====================

If more evidence is needed:

{{
    "status": "need_more_evidence",
    "reasoning": "Brief explanation of what the current evidence suggests and why more is needed.",
    "current_hypotheses": [
        "Most likely hypothesis",
        "Second hypothesis (if still plausible)"
    ],
    "next_action": {{
        "tool": "<tool_name>",
        "arguments": {{
            "pod_name": "<value>",
            "namespace": "<value>"
        }}
    }},
    "root_cause": null,
    "recovery_plan": []
}}

If the investigation is complete:

{{
    "status": "complete",
    "reasoning": "Explain why the evidence uniquely supports this root cause and why competing hypotheses were eliminated.",
    "current_hypotheses": [
        "Surviving root cause hypothesis"
    ],
    "next_action": null,
    "root_cause": "One-sentence description of the confirmed root cause.",
    "recovery_plan": [
        "Step 1: ...",
        "Step 2: ..."
    ]
}}

Return JSON only. No extra text before or after the JSON.
"""


## `discovery.py`
Pure Python cluster discovery — no LLM involved. Scans the cluster (via the remote tool server) and returns unhealthy workloads. Unaffected by the model swap.


In [7]:
"""
discovery.py

Pure Python cluster discovery — no LLM involved.

Responsibility:
    Scan the cluster, detect unhealthy workloads, return them
    as structured dicts so the investigation phase knows what to investigate.
"""

# NOTE: discover_unhealthy_workloads is already defined above in this notebook.


def run_discovery(namespace: str = "--all-namespaces") -> tuple[list[dict], int]:
    """
    Scan the cluster and return (unhealthy_workloads, total_pod_count).

    unhealthy_workloads is a list of dicts:
    [
        {
            "pod_name": "payment-abc123",
            "namespace": "devdas",
            "status": "CrashLoopBackOff",
            "restart_count": 17,
            "node": "kind-worker",
            "owner_kind": "Deployment",
            "owner_name": "payment-service"
        },
        ...
    ]
    """
    print("\n" + "=" * 60)
    print("  DevDas — Discovery Phase")
    print("=" * 60)
    print(f"  Scanning namespace(s): {namespace}")

    unhealthy, total_pods = discover_unhealthy_workloads(namespace)

    healthy_count = total_pods - len(unhealthy)

    print(f"\n  Total pods scanned  : {total_pods}")
    print(f"  Healthy pods        : {healthy_count}")
    print(f"  Unhealthy pods      : {len(unhealthy)}")

    if unhealthy:
        print("\n  Unhealthy workloads detected:")
        for w in unhealthy:
            print(f"    • [{w['namespace']}] {w['pod_name']}  →  {w['status']}")
    else:
        print("\n  ✅ All pods appear healthy. Nothing to investigate.")

    print("=" * 60)

    return unhealthy, total_pods


## `investigate.py`
Per-workload investigation orchestrator — runs the LLM reasoning loop (now backed by local Qwen via `ask_llm()`) with tool calls, plus deterministic regex pattern shortcuts that skip the LLM entirely for definitive root causes (e.g. missing ConfigMap/Secret, ImagePullBackOff).


In [8]:
"""
investigate.py

Per-workload investigation orchestrator.

Responsibilities:
    - Accept a single unhealthy workload dict from the discovery phase.
    - Create an isolated per-workload evidence dict.
    - Run the LLM reasoning loop (preserved from original architecture).
    - Return a structured investigation result.

The reasoning loop:
    1. Build a prompt with workload context + collected evidence.
    2. Ask the LLM: "what tool next, or are you done?"
    3. If "need_more_evidence" → execute the requested tool → add to evidence → repeat.
    4. If "complete" → return the final diagnosis.
    5. Safeguards: repeated tool requests, invalid tool names, max iterations.
"""

# NOTE: new_evidence, AVAILABLE_TOOLS, execute_tool, ask_llm, and resolve_deployment_name
# are already defined above in this notebook.
import json
import time
import re



# Maximum iterations before forcing a final answer
MAX_ITERATIONS = 20
DEFINITIVE_PATTERNS = [
    (r'configmap\s+"([^"]+)"\s+not found', "Missing ConfigMap: '{0}' does not exist."),
    (r'secret\s+"([^"]+)"\s+not found', "Missing Secret: '{0}' does not exist."),
    (r'ErrImageNeverPull', "Image pull policy misconfiguration (ErrImageNeverPull)."),
    (r'ImagePullBackOff', "Image cannot be pulled — check image name/tag/registry access."),
    (r'ErrImagePull', "Image pull failure — check image name/tag/registry access."),
    (r'FailedScheduling.*Insufficient (cpu|memory)', "Pod unschedulable — insufficient {0} on cluster nodes."),
    (r'FailedScheduling.*node\(s\) had taint', "Pod unschedulable — no node tolerates required taints."),
]

def _recovery_plan_for(root_cause: str) -> list[str]:
    """Map a definitive root cause to a concrete recovery plan."""
    rc = root_cause.lower()

    if "missing configmap" in rc:
        return [
            "Identify the correct ConfigMap manifest for this workload (check git history or Helm chart).",
            "Recreate the ConfigMap in the correct namespace: kubectl apply -f <configmap.yaml>",
            "Verify the pod restarts and reaches Running state.",
        ]
    if "missing secret" in rc:
        return [
            "Identify the correct Secret source (vault, sealed-secret, or CI/CD pipeline).",
            "Recreate the Secret in the correct namespace.",
            "Verify the pod restarts and reaches Running state.",
        ]
    if "errimageneverpull" in rc or "image pull policy" in rc:
        return [
            "Confirm the image exists locally on the node, or",
            "Set imagePullPolicy to IfNotPresent/Always and ensure registry access is configured.",
        ]
    if "image cannot be pulled" in rc or "image pull failure" in rc:
        return [
            "Verify the image name/tag is correct in the deployment spec.",
            "Confirm registry credentials (imagePullSecrets) are valid.",
            "Manually pull the image to confirm registry accessibility.",
        ]
    if "unschedulable" in rc and "cpu" in rc.lower():
        return [
            "Check node resource availability (kubectl describe nodes).",
            "Reduce pod resource requests, or scale/add cluster nodes.",
        ]
    if "unschedulable" in rc and "taint" in rc.lower():
        return [
            "Add matching tolerations to the pod spec, or",
            "Schedule on nodes without the conflicting taint.",
        ]

    return ["Manual investigation required — no standard recovery plan matched."]

def detect_definitive_cause(evidence: dict):
    for tool_name, result in evidence.items():
        if not isinstance(result, str):
            continue
        for pattern, template in DEFINITIVE_PATTERNS:
            m = re.search(pattern, result, re.IGNORECASE)
            if m:
                return template.format(*m.groups())
    return None

def investigate(workload: dict, system_prompt: str) -> dict:
    """
    Investigate a single unhealthy Kubernetes workload.

    Args:
        workload:      Dict with keys: pod_name, namespace, status,
                       restart_count, node, owner_kind, owner_name
        system_prompt: The DevDas SRE system prompt string

    Returns:
        A result dict containing:
            workload, root_cause, recovery_plan, reasoning,
            tools_used (list), iterations, duration_seconds, status
    """
    pod_name      = workload["pod_name"]
    namespace     = workload["namespace"]
    status        = workload["status"]
    owner_kind    = workload["owner_kind"]
    owner_name    = workload["owner_name"]
    restart_count = workload["restart_count"]

    # ── Build the workload incident description ────────────────────────────
    incident_description = (
        f"Pod Name      : {pod_name}\n"
        f"Namespace     : {namespace}\n"
        f"Status        : {status}\n"
        f"Restart Count : {restart_count}\n"
        f"Owner         : {owner_kind}/{owner_name}\n"
    )
    evidence: dict = new_evidence()
    deployment_name = resolve_deployment_name(pod_name, namespace) if owner_kind == "ReplicaSet" else owner_name
    evidence["_resolved_deployment_name"] = deployment_name  # add this
    incident_description += f"Deployment    : {deployment_name}\n"

    # ── Per-workload isolated state ────────────────────────────────────────
    used_tools: set = set()

    iteration: int = 0
    repeated_requests: int = 0
    start_time = time.time()    
    repeated_requests: int = 0
    start_time = time.time()


    print(f"\n{'─' * 60}")
    print(f"  Investigating: {pod_name}  [{namespace}]")
    print(f"  Status       : {status}")
    print(f"{'─' * 60}")

    # ── Reasoning loop ─────────────────────────────────────────────────────
    while iteration < MAX_ITERATIONS:
        iteration += 1
        print(f"\n  ── Iteration {iteration} ──")

        # Summary of current investigation state
        investigation_summary = {
            "tools_used": sorted(list(used_tools)),
            "evidence_keys_collected": sorted([
                k for k, v in evidence.items() if v is not None
            ])
        }

        # Build the available tools description (exclude already-used tools)
        remaining_tools = {
            name: meta
            for name, meta in AVAILABLE_TOOLS.items()
            if name not in used_tools
        }
        tools_description = json.dumps(
            {name: meta["description"] for name, meta in remaining_tools.items()},
            indent=2
        )

        # Build the full prompt for this iteration
        prompt = f"""
Workload Under Investigation:

{incident_description}

Collected Evidence:

{json.dumps(evidence, indent=2)}

Investigation Summary:

{json.dumps(investigation_summary, indent=2)}

Remaining Available Tools (unused):

{tools_description}

Based ONLY on the workload description and collected evidence,
decide whether you can reach a confident conclusion.

If yes  → return status: "complete" with root_cause and recovery_plan.
If no   → request EXACTLY ONE tool from the remaining available tools list.
         Include ALL required arguments for that tool.
"""

        # ── Ask the LLM ───────────────────────────────────────────────────
        response_raw = ask_llm(system_prompt, prompt)

        print(f"\n  LLM Response:")
        print("  " + response_raw.replace("\n", "\n  "))

        # ── Parse JSON ────────────────────────────────────────────────────
        try:
            data = json.loads(response_raw)
        except json.JSONDecodeError as e:
            print(f"\n  [WARNING] JSON parse error: {e}. Retrying...")
            evidence["SYSTEM_PARSE_ERROR"] = f"LLM returned invalid JSON: {response_raw[:200]}"
            continue

        # ── Check if investigation is complete ────────────────────────────
        if data.get("status") == "complete":
            duration = round(time.time() - start_time, 1)
            print(f"\n  ✅ Investigation complete in {iteration} iteration(s), {duration}s")

            return {
                "workload": workload,
                "status": "complete",
                "root_cause": data.get("root_cause", "Unknown"),
                "recovery_plan": data.get("recovery_plan", []),
                "reasoning": data.get("reasoning", ""),
                "current_hypotheses": data.get("current_hypotheses", []),
                "tools_used": sorted(list(used_tools)),
                "iterations": iteration,
                "repeated_requests": repeated_requests,
                "duration_seconds": duration,
            }

        # ── Validate next_action ──────────────────────────────────────────
        next_action = data.get("next_action")
        if not next_action:
            evidence["SYSTEM_WARNING"] = "LLM returned need_more_evidence but no next_action."
            continue

        tool = next_action.get("tool", "")
        args = next_action.get("arguments", {})

        # ── Invalid tool name ──────────────────────────────────────────────
        if tool not in AVAILABLE_TOOLS:
            evidence["SYSTEM_ERROR"] = {
                "error": f"'{tool}' is not a valid tool.",
                "valid_tools": list(AVAILABLE_TOOLS.keys())
            }
            print(f"  [WARNING] LLM requested invalid tool: '{tool}'")
            continue

        # ── Tool already used ──────────────────────────────────────────────
        if tool in used_tools:
            repeated_requests += 1
            remaining = [t for t in AVAILABLE_TOOLS if t not in used_tools]

            if not remaining:
                # All tools exhausted — force a final answer
                print("\n  [INFO] All tools exhausted. Requesting forced final diagnosis.")
                return _force_final_diagnosis(
                    system_prompt, incident_description, evidence,
                    used_tools, iteration, repeated_requests, start_time, workload
                )

            evidence["SYSTEM_WARNING"] = {
                "warning": f"Tool '{tool}' was already used. Please pick a different tool.",
                "remaining_tools": remaining
            }
            print(f"  [WARNING] Tool '{tool}' already used. Nudging LLM.")
            continue
        # ---------------------------------------------------------------
        # Execute the selected tool
        # ---------------------------------------------------------------

        print(f"\n  Running Tool -> {tool}")

        try:
            result = execute_tool(tool, args, evidence)

        except Exception as e:
            evidence["SYSTEM_ERROR"] = {
                "error": str(e),
                "tool": tool,
            }
            print(f"\n  [ERROR] Tool execution failed: {e}")
            continue

        used_tools.add(tool)

        preview = result

        if isinstance(preview, str) and len(preview) > 1200:
            preview = preview[:1200] + "\n\n... (truncated)"

        print("\n  Tool Result:")
        print("  " + str(preview).replace("\n", "\n  "))

        evidence[tool] = result

        definitive = detect_definitive_cause(evidence)
        if definitive:
            duration = round(time.time() - start_time, 1)
            return {
                "workload": workload,
                "status": "complete",
                "root_cause": definitive,
                "recovery_plan": _recovery_plan_for(definitive),  # small lookup table, see below
                "reasoning": "Definitive Kubernetes evidence matched a known failure signature; no further investigation needed.",
                "current_hypotheses": [definitive],
                "tools_used": sorted(list(used_tools)),
                "iterations": iteration,
                "repeated_requests": repeated_requests,
                "duration_seconds": duration,
            }



def _force_final_diagnosis(
    system_prompt, incident_description, evidence,
    used_tools, iteration, repeated_requests, start_time, workload
) -> dict:
    """
    Force the LLM to produce a final diagnosis when tools are exhausted
    or max iterations are reached.
    """
    final_prompt = f"""
Workload Under Investigation:

{incident_description}

Collected Evidence:

{json.dumps(evidence, indent=2)}

All available tools have been exhausted or max iterations reached.
You MUST return status: "complete" now.
Produce the best possible diagnosis from the evidence you have.
Do not request any more tools.
"""
    response_raw = ask_llm(system_prompt, final_prompt)

    try:
        data = json.loads(response_raw)
    except json.JSONDecodeError:
        data = {
            "root_cause": "Unable to determine — LLM returned invalid JSON.",
            "recovery_plan": ["Manual investigation required."],
            "reasoning": response_raw[:500],
        }

    duration = round(time.time() - start_time, 1)

    return {
        "workload": workload,
        "status": "complete_forced",
        "root_cause": data.get("root_cause", "Unknown"),
        "recovery_plan": data.get("recovery_plan", []),
        "reasoning": data.get("reasoning", ""),
        "current_hypotheses": data.get("current_hypotheses", []),
        "tools_used": sorted(list(used_tools)),
        "iterations": iteration,
        "repeated_requests": repeated_requests,
        "duration_seconds": duration,
    }

## `report.py`
Formats and prints the final Cluster Investigation Report. Purely presentational — no LLM or model-specific logic.


In [9]:
"""
report.py

Cluster Investigation Report generator.

Formats and prints the final cluster health report after all
per-workload investigations are complete.
"""

import time


# ── ANSI color helpers (graceful fallback on Windows without color support) ──

def _bold(text: str) -> str:
    return f"\033[1m{text}\033[0m"

def _green(text: str) -> str:
    return f"\033[92m{text}\033[0m"

def _yellow(text: str) -> str:
    return f"\033[93m{text}\033[0m"

def _red(text: str) -> str:
    return f"\033[91m{text}\033[0m"

def _cyan(text: str) -> str:
    return f"\033[96m{text}\033[0m"


def generate_report(
    results: list[dict],
    total_pods: int,
    unhealthy_count: int,
    cluster_start_time: float,
) -> str:
    """
    Generate and print the Cluster Investigation Report.

    Args:
        results:           List of investigation result dicts from investigate()
        total_pods:        Total pods scanned during discovery
        unhealthy_count:   Number of unhealthy pods found
        cluster_start_time: time.time() when the overall run started

    Returns:
        The report as a plain-text string (also printed to stdout).
    """
    healthy_count    = total_pods - unhealthy_count
    total_duration   = round(time.time() - cluster_start_time, 1)
    total_tool_calls = sum(len(r["tools_used"]) for r in results)
    avg_tool_calls   = (
        round(total_tool_calls / len(results), 1) if results else 0
    )
    total_iterations = sum(r["iterations"] for r in results)

    lines = []

    def line(text=""):
        lines.append(text)
        print(text)

    # ── Header ──────────────────────────────────────────────────────────────
    line()
    line(_bold("╔" + "═" * 58 + "╗"))
    line(_bold("║" + "   DevDas — Cluster Investigation Report".center(58) + "║"))
    line(_bold("╚" + "═" * 58 + "╝"))
    line()

    # ── Cluster Summary ──────────────────────────────────────────────────────
    line(_bold("  Cluster Summary"))
    line("  " + "─" * 40)
    line(f"  {'Total Pods Scanned':<28}: {total_pods}")
    line(f"  {'Unhealthy Workloads Investigated':<28}: {_red(str(unhealthy_count)) if unhealthy_count else _green('0')}")
    line(f"  {'Healthy Pods':<28}: {_green(str(healthy_count))}")
    line(f"  {'Total Investigations':<28}: {len(results)}")
    line(f"  {'Total Tool Calls':<28}: {total_tool_calls}")
    line(f"  {'Total Iterations':<28}: {total_iterations}")
    line(f"  {'Avg Tool Calls / Investigation':<28}: {avg_tool_calls}")
    line(f"  {'Total Duration':<28}: {total_duration}s")
    line()

    if not results:
        line(_green("  ✅ No unhealthy workloads found. Cluster appears healthy."))
        line()
        report_text = "\n".join(lines)
        return report_text

    # ── Per-Workload Results ─────────────────────────────────────────────────
    line(_bold("  Investigation Results"))
    line("  " + "─" * 56)

    for idx, result in enumerate(results, start=1):
        wl       = result["workload"]
        pod      = wl["pod_name"]
        ns       = wl["namespace"]
        status   = wl["status"]
        owner    = f"{wl['owner_kind']}/{wl['owner_name']}"
        restarts = wl["restart_count"]

        root_cause    = result.get("root_cause", "Unknown")
        recovery_plan = result.get("recovery_plan", [])
        tools_used    = result.get("tools_used", [])
        iterations    = result.get("iterations", 0)
        duration      = result.get("duration_seconds", 0)
        inv_status    = result.get("status", "complete")

        status_badge = (
            _yellow("⚠ complete_forced")
            if inv_status == "complete_forced"
            else _green("✅ complete")
        )

        line()
        line(f"  [{idx}] {_bold(pod)}  ({ns})")
        line(f"       Pod Status   : {_red(status)}")
        line(f"       Owner        : {owner}")
        line(f"       Restarts     : {restarts}")
        line(f"       Inv. Status  : {status_badge}")
        line(f"       Tool Calls   : {len(tools_used)}")
        line(f"       Iterations   : {iterations}")
        line(f"       Duration     : {duration}s")
        line()
        line(f"       {_bold('Root Cause')}")
        line(f"       {root_cause}")
        line()

        if recovery_plan:
            line(f"       {_bold('Recovery Plan')}")
            for step_num, step in enumerate(recovery_plan, start=1):
                line(f"       {step_num}. {step}")
        else:
            line(f"       {_yellow('Recovery plan: None provided.')}")

        if tools_used:
            line()
            line(f"       {_bold('Tools Used')}  : {', '.join(tools_used)}")

        line()
        line("  " + "─" * 56)

    # ── Footer ───────────────────────────────────────────────────────────────
    line()
    line(_cyan("  DevDas investigation complete."))
    line(_cyan("  This report was generated autonomously by the DevDas SRE agent."))
    line()

    report_text = "\n".join(lines)
    return report_text


def save_report(report_text: str, output_path: str = "devdas_report.txt"):
    """Save the plain-text report to a file (ANSI codes stripped)."""
    import re
    ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
    clean_text = ansi_escape.sub('', report_text)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(clean_text)
    print(f"\n  Report saved to: {output_path}")


## `system_prompt` instance
Instantiate the system prompt now that `AVAILABLE_TOOLS` is defined (mirrors `system_prompt.py`'s module-level execution). This is the text sent as the `system` message to Qwen on every `ask_llm()` call.


In [10]:
# system_prompt.py builds `system_prompt` at import time using AVAILABLE_TOOLS.
# Re-run the same construction here now that AVAILABLE_TOOLS exists in this notebook.

_tool_list = "\n".join(
    f"  - {name}: {meta['description']}"
    for name, meta in AVAILABLE_TOOLS.items()
)


## `main.py` — CLI argument parsing
Defines `parse_args()` for the standalone-script version of DevDas (`python main.py --namespace ... --save-report`). **Not used when running from this notebook** — Jupyter injects its own kernel launcher flags into `sys.argv`, which breaks `argparse`. Use the notebook-friendly `run(...)` function below instead; this cell is kept only for parity with the original script.


In [12]:
def parse_args():
    parser = argparse.ArgumentParser(
        description="DevDas — Autonomous Kubernetes SRE Investigation Agent"
    )
    parser.add_argument(
        "--namespace", "-n",
        default="--all-namespaces",
        help="Kubernetes namespace to scan. Default: --all-namespaces"
    )
    parser.add_argument(
        "--save-report",
        action="store_true",
        help="Save the investigation report to devdas_report.txt"
    )
    return parser.parse_args()


### Run the pipeline
Call `run(...)` directly with explicit arguments — this is the notebook-friendly entry point and bypasses CLI arg parsing, which doesn't work in Jupyter (see the "how to run" notes if you hit an `argparse`/`sys.argv` error from `main()`).

Requires: the Qwen `tokenizer`/`model` loaded (top of notebook), and a reachable Kubernetes cluster through the local tool server + ngrok tunnel. No API key needed anymore.


In [ ]:
def run(namespace: str = "--all-namespaces", write_report_file: bool = False):
    """Notebook-friendly entry point equivalent to running `python main.py`."""
    cluster_start_time = time.time()

    print(BANNER)
    print(f"  Starting cluster investigation...")
    print(f"  Namespace scope : {namespace}")
    print()

    # Phase 1: Discovery
    unhealthy_workloads, total_pods = run_discovery(namespace=namespace)
    unhealthy_workloads.sort(key=lambda w: w["owner_name"])
    results = []
    for owner_name, group in groupby(unhealthy_workloads, key=lambda w: w["owner_name"]):
        group = list(group)
        representative = group[0]
        result = investigate(representative, system_prompt)
        results.append(result)
        for sibling in group[1:]:
            sibling_result = dict(result)
            sibling_result["workload"] = sibling
            sibling_result["reasoning"] += " (Same root cause as sibling pod under identical owner.)"
            results.append(sibling_result)

    if not unhealthy_workloads:
        print("\n  \u2705 Cluster is healthy. No investigations needed.\n")
        generate_report(
            results=[],
            total_pods=total_pods,
            unhealthy_count=0,
            cluster_start_time=cluster_start_time,
        )
        return

    print(f"\n  Found {len(unhealthy_workloads)} unhealthy workload(s).")
    print(f"  Starting per-workload investigations...\n")

    # Phase 3: Report
    print(f"\n\n{'=' * 60}")
    print(f"  Generating Cluster Investigation Report...")
    print(f"{'=' * 60}\n")

    report_text = generate_report(
        results=results,
        total_pods=total_pods,
        unhealthy_count=len(unhealthy_workloads),
        cluster_start_time=cluster_start_time,
    )

    if write_report_file:
        save_report(report_text)

    return

# Uncomment to run against a live cluster:
# run(namespace="--all-namespaces", write_report_file=False)
run(namespace="devdas", write_report_file=False)
